# RVC — Hướng Dẫn Training Giọng Nói Từng Bước Chi Tiết

**Notebook này dành cho người mới**, giải thích rõ từng tham số và chia nhỏ từng bước để dễ theo dõi.

## Tổng quan pipeline

```
[File .wav gốc]
       ↓
  BƯỚC 0: Kiểm tra môi trường
       ↓
  BƯỚC 1: Chuẩn bị dataset
       ↓
  BƯỚC 2: Khởi tạo Python
       ↓
  BƯỚC 3: Đặt tham số
       ↓
  BƯỚC 4: Preprocess — cắt, chuẩn hóa audio
       ↓
  BƯỚC 5: Trích F0 — phân tích cao độ giọng
       ↓
  BƯỚC 6: Trích đặc trưng Hubert — "nội dung" giọng
       ↓
  BƯỚC 7: Huấn luyện G + D (lâu nhất)
       ↓
  BƯỚC 8: Tạo FAISS Index (tùy chọn)
       ↓
  BƯỚC 9: Xuất file .pth nhỏ cho inference
       ↓
[G_*.pth + added_*.index + model_infer.pth]
```

**Yêu cầu working directory:** mở notebook này từ thư mục `rvc_standalone` (phải thấy `infer/`, `configs/`, `training_pipeline/`).

## Yêu cầu trước khi bắt đầu

| Yêu cầu | Lệnh cài / kiểm tra |
|---------|---------------------|
| Cài thư viện Python | `pip install -r requirements.txt` trong `rvc_standalone` |
| Tải model weights | `python tools/download_assets.py` |
| FFmpeg trên PATH | `ffmpeg -version` trong terminal |
| `logs/mute/` tồn tại | Sẽ kiểm tra ở Bước 0 |
| GPU với CUDA (khuyến nghị) | CPU được nhưng rất chậm |

**Thư mục model weights cần có:**
- `assets/hubert/hubert_base.pt` — dùng ở Bước 6
- `assets/rmvpe/rmvpe.pt` — dùng ở Bước 5 (nếu dùng rmvpe)
- `assets/pretrained_v2/f0G40k.pth` và `f0D40k.pth` — pretrained cho Bước 7

---
# BƯỚC 0: Kiểm Tra Môi Trường

**Chạy toàn bộ phần này trước khi làm gì khác.** Đảm bảo mọi thứ sẵn sàng.

### 0.1 — Kiểm tra Working Directory

Notebook phải chạy từ thư mục `rvc_standalone`. Nếu sai, Jupyter sẽ không tìm thấy các file cần thiết.

In [4]:
import pathlib

CWD = pathlib.Path.cwd().resolve()
print("Working directory hiện tại:", CWD)

# Kiểm tra các thư mục bắt buộc
cac_thu_muc_can_co = [
    "infer",
    "configs",
    "training_pipeline",
    "assets",
    "logs",
]

thieu = []
for thu_muc in cac_thu_muc_can_co:
    ton_tai = (CWD / thu_muc).is_dir()
    trang_thai = "✓" if ton_tai else "✗ THIẾU"
    print(f"  {trang_thai}  {thu_muc}/")
    if not ton_tai:
        thieu.append(thu_muc)

if thieu:
    print("\n❌ Thiếu thư mục:", thieu)
    print("   → Mở Jupyter từ thư mục rvc_standalone, không phải thư mục cha.")
else:
    print("\n✅ Working directory OK — đang ở rvc_standalone")

Working directory hiện tại: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\compare
  ✓  infer/
  ✓  configs/
  ✓  training_pipeline/
  ✓  assets/
  ✓  logs/

✅ Working directory OK — đang ở rvc_standalone


### 0.2 — Kiểm tra GPU và CUDA

RVC train rất chậm trên CPU. Nếu không có GPU, training 100 epoch có thể mất hàng chục giờ.

In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Số GPU:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f"  GPU {i}: {props.name}  |  VRAM: {vram_gb:.1f} GB")
    print("\n✅ GPU sẵn sàng. Training sẽ chạy trên CUDA.")
else:
    print("\n⚠️  Không tìm thấy GPU CUDA.")
    print("   Training vẫn chạy được trên CPU nhưng rất chậm (~10-50x).")
    print("   Khuyến nghị: dùng Google Colab hoặc Kaggle nếu không có GPU.")

PyTorch version: 2.6.0+cu124
CUDA available: True
Số GPU: 1
  GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU  |  VRAM: 4.0 GB

✅ GPU sẵn sàng. Training sẽ chạy trên CUDA.


### 0.3 — Kiểm tra Model Weights (Hubert, RMVPE, Pretrained G/D)

Nếu thiếu file nào, chạy `python tools/download_assets.py` trong terminal.

In [7]:
import pathlib

CWD = pathlib.Path.cwd().resolve()

# Danh sách file weights cần có
weights_can_co = [
    ("assets/hubert/hubert_base.pt",       "Hubert — trích đặc trưng giọng (Bước 6)"),
    ("assets/rmvpe/rmvpe.pt",              "RMVPE — phân tích F0 (Bước 5, nếu dùng rmvpe)"),
    ("assets/pretrained_v2/f0G40k.pth",   "Pretrained Generator 40k (Bước 7)"),
    ("assets/pretrained_v2/f0D40k.pth",   "Pretrained Discriminator 40k (Bước 7)"),
    ("assets/pretrained_v2/f0G48k.pth",   "Pretrained Generator 48k (chỉ cần nếu dùng 48k)"),
    ("assets/pretrained_v2/f0D48k.pth",   "Pretrained Discriminator 48k (chỉ cần nếu dùng 48k)"),
]

print("Kiểm tra model weights:\n")
thieu = []
for duong_dan, mo_ta in weights_can_co:
    f = CWD / duong_dan
    if f.exists():
        size_mb = f.stat().st_size / 1024**2
        print(f"  ✓  {duong_dan} ({size_mb:.0f} MB)")
        print(f"       → {mo_ta}")
    else:
        print(f"  ✗  {duong_dan} — THIẾU")
        print(f"       → {mo_ta}")
        thieu.append(duong_dan)

print()
if thieu:
    print(f"❌ Thiếu {len(thieu)} file. Chạy: python tools/download_assets.py")
else:
    print("✅ Tất cả weights có đủ.")

# Kiểm tra thêm logs/mute
mute_dir = CWD / "logs" / "mute"
print(f"\nlogs/mute/: {'✓ OK' if mute_dir.exists() else '✗ THIẾU — sẽ báo lỗi ở Bước 7'}")

Kiểm tra model weights:

  ✗  assets/hubert/hubert_base.pt — THIẾU
       → Hubert — trích đặc trưng giọng (Bước 6)
  ✗  assets/rmvpe/rmvpe.pt — THIẾU
       → RMVPE — phân tích F0 (Bước 5, nếu dùng rmvpe)
  ✗  assets/pretrained_v2/f0G40k.pth — THIẾU
       → Pretrained Generator 40k (Bước 7)
  ✗  assets/pretrained_v2/f0D40k.pth — THIẾU
       → Pretrained Discriminator 40k (Bước 7)
  ✗  assets/pretrained_v2/f0G48k.pth — THIẾU
       → Pretrained Generator 48k (chỉ cần nếu dùng 48k)
  ✗  assets/pretrained_v2/f0D48k.pth — THIẾU
       → Pretrained Discriminator 48k (chỉ cần nếu dùng 48k)

❌ Thiếu 6 file. Chạy: python tools/download_assets.py

logs/mute/: ✓ OK


In [8]:
!python tools/download_assets.py

BASE_DIR = D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\project\compare
HF base URL = https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/
  [!] Lỗi tải (lần 1/6): ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
      Chờ 8s rồi thử lại...
  [!] Lỗi tải (lần 2/6): ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
      Chờ 16s rồi thử lại...
  D32k.pth...
  D40k.pth...
  D48k.pth...
  G32k.pth...
  G40k.pth...
  G48k.pth...
  f0D32k.pth...
  f0D40k.pth...
  f0D48k.pth...
  f0G32k.pth...
  f0G40k.pth...
  f0G48k.pth...
  D32k.pth...
  D40k.pth...
  D48k.pth...
  G32k.pth...
  G40k.pth...
  G48k.pth...
  f0D32k.pth...
  f0D40k.pth...
  f0D48k.pth...
  f0G32k.pth...
  f0G40k.pth...
  f0G48k.pth...
  HP2-%E4%BA%BA%E5%A3%B0vocals%2B%E9%9D%9E%E4%BA%BA%E5%A3%B0instrumentals.pth...
  HP2_all_vocals.pth...
  HP3_

---
# BƯỚC 1: Chuẩn Bị Dataset (File .wav của bạn)

**Đây là giọng mà model sẽ học để bắt chước.** Sau khi train xong, khi bạn infer một bài hát, giọng trong bài sẽ được chuyển sang giống giọng này.

### 1.1 — Yêu cầu về chất lượng audio

| Tiêu chí | Yêu cầu tối thiểu | Khuyến nghị |
|----------|-------------------|-------------|
| Định dạng | `.wav` | `.wav` 16-bit hoặc 32-bit float |
| Sample rate | Bất kỳ (sẽ được resample) | 44.1kHz hoặc 48kHz |
| Nội dung | Giọng nói/hát rõ ràng | Hát, ít nhạc nền |
| Thời lượng tổng | ≥ 5 phút | 10–30 phút |
| Nền ồn | Càng ít càng tốt | Phòng yên tĩnh |
| Số file | ≥ 1 | Nhiều file ngắn (~3-10 phút/file) |

**Lưu ý:**
- Nếu audio có nhạc nền, hãy dùng vocal separator trước (ví dụ: UVR, Demucs)
- Không cần cắt tay — bước Preprocess sẽ tự cắt đoạn im lặng
- Tránh dấu cách trong tên file/thư mục

### 1.2 — Tạo thư mục và đặt file .wav vào

In [ ]:
import pathlib

# ============================================================
# SỬA ĐÂY: Đặt tên thư mục cho dataset của bạn
# Không dùng dấu cách, dấu tiếng Việt, ký tự đặc biệt
# Ví dụ: "giong_nguyen_van_a", "voice_male_01", "singer_pop"
# ============================================================
TEN_DATASET = "giong_cua_toi"  # <-- SỬA TÊN NÀY

dataset_dir = pathlib.Path("datasets") / TEN_DATASET
dataset_dir.mkdir(parents=True, exist_ok=True)

print("Thư mục dataset đã tạo:")
print(" ", dataset_dir.resolve())
print()
print("Bây giờ hãy COPY các file .wav của bạn vào thư mục trên.")
print("Sau khi copy xong, chạy ô 1.3 để kiểm tra.")

### 1.3 — Kiểm tra file .wav đã copy vào chưa

Chạy ô này SAU KHI đã copy file .wav vào thư mục.

In [ ]:
import pathlib

dataset_dir = pathlib.Path("datasets") / TEN_DATASET  # dùng biến từ ô 1.2

wavs = sorted(list(dataset_dir.glob("*.wav")) + list(dataset_dir.glob("*.WAV")))

if not wavs:
    print("❌ Không tìm thấy file .wav trong:", dataset_dir.resolve())
    print("   Hãy copy file .wav vào thư mục đó rồi chạy lại ô này.")
else:
    print(f"✅ Tìm thấy {len(wavs)} file .wav:\n")
    tong_size = 0
    for f in wavs:
        size_mb = f.stat().st_size / 1024**2
        tong_size += size_mb
        print(f"  {f.name:40s}  {size_mb:6.1f} MB")
    print(f"\n  Tổng: {tong_size:.1f} MB")

### 1.4 — Xem thông tin chi tiết audio (sample rate, thời lượng)

Ô này dùng `librosa` để xem thông tin từng file. Hữu ích để biết tổng thời lượng dataset.

In [ ]:
import pathlib
import warnings
warnings.filterwarnings("ignore")

try:
    import librosa
    co_librosa = True
except ImportError:
    co_librosa = False
    print("librosa chưa cài. Bỏ qua ô này.")

if co_librosa:
    dataset_dir = pathlib.Path("datasets") / TEN_DATASET
    wavs = sorted(list(dataset_dir.glob("*.wav")) + list(dataset_dir.glob("*.WAV")))

    if not wavs:
        print("Chưa có file .wav. Chạy ô 1.3 trước.")
    else:
        tong_giay = 0
        print(f"{'File':<40} {'Sample Rate':>12} {'Thời lượng':>12} {'Kênh':>6}")
        print("-" * 75)
        for f in wavs:
            try:
                y, sr = librosa.load(str(f), sr=None, mono=False)
                if y.ndim == 1:
                    kenh = 1
                    thoi_luong = len(y) / sr
                else:
                    kenh = y.shape[0]
                    thoi_luong = y.shape[1] / sr
                tong_giay += thoi_luong
                phut = int(thoi_luong // 60)
                giay = thoi_luong % 60
                print(f"{f.name:<40} {sr:>12,} Hz {phut:>4}:{giay:05.2f} min {kenh:>6}")
            except Exception as e:
                print(f"{f.name:<40} Lỗi đọc: {e}")

        print("-" * 75)
        tong_phut = int(tong_giay // 60)
        tong_giay_le = tong_giay % 60
        print(f"Tổng thời lượng: {tong_phut} phút {tong_giay_le:.0f} giây")

        if tong_giay < 300:
            print("\n⚠️  Dataset ngắn (< 5 phút). Model có thể kém chất lượng.")
            print("   Khuyến nghị: >= 10 phút audio sạch.")
        elif tong_giay < 600:
            print("\n⚠️  Dataset tạm ổn (5-10 phút). Thêm audio nếu có thể.")
        else:
            print("\n✅ Thời lượng dataset tốt.")

---
# BƯỚC 2: Khởi Tạo Môi Trường Python

**Chạy một lần** sau khi mở notebook hoặc sau khi Restart kernel.

Bước này:
1. Đặt working directory về `rvc_standalone`
2. Load module `training_pipeline` vào `sys.path`
3. Tạo `config` — chứa thông tin về GPU, Python interpreter, FP16...

> Sau khi Restart Kernel, luôn chạy lại ô này trước khi làm bất cứ điều gì khác.

In [ ]:
import logging
import os
import pathlib
import sys

# Đặt working dir về rvc_standalone
STANDALONE_ROOT = pathlib.Path.cwd().resolve()

# Kiểm tra đang đúng thư mục
if not (STANDALONE_ROOT / "infer" / "modules" / "train" / "train.py").is_file():
    raise SystemExit(
        "\n❌ Working directory sai!"
        "\n   Phải chạy từ thư mục rvc_standalone."
        "\n   Trong Jupyter: File → Open → chọn thư mục rvc_standalone."
    )

os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

# Import các module của training pipeline
from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

# bootstrap() tự phát hiện GPU, Python interpreter, cài đặt FP16
root, config = bootstrap()
assert root == STANDALONE_ROOT

print("=" * 50)
print("Thư mục gốc :", STANDALONE_ROOT)
print("Python      :", config.python_cmd)
print("Device Hubert:", config.device)      # cuda:0 hoặc cpu
print("Half-prec   :", config.is_half)      # FP16 (True nếu có GPU hỗ trợ)
print("=" * 50)
print("✅ Bước 2 xong — sẵn sàng cấu hình tham số")

---
# BƯỚC 3: Cấu Hình Tham Số Training

Đây là phần quan trọng nhất. Bạn cần đặt đúng các tham số trước khi train.

### 3.1 — Giải thích chi tiết từng tham số

#### Tham số nhận diện thí nghiệm

| Tham số | Giải thích | Ví dụ |
|---------|-----------|-------|
| `experiment_name` | Tên thư mục chứa toàn bộ output trong `logs/`. Đặt tên dễ nhớ, không dấu cách | `"giong_A"`, `"nguyen_van_a_v1"` |
| `trainset_dir` | Thư mục chứa file .wav gốc (tương đối so với rvc_standalone) | `"datasets/giong_cua_toi"` |

---

#### Tham số về chất lượng âm thanh

| Tham số | Giải thích | Nên chọn gì? |
|---------|-----------|----------------|
| `sample_rate_label` | Sample rate train: `"32k"`, `"40k"`, hoặc `"48k"` | **`"40k"`** cho giọng nói/hát thông thường. `"48k"` cho chất lượng cao hơn nhưng cần nhiều VRAM hơn |
| `version` | `"v1"` (feature 256 chiều) hoặc `"v2"` (feature 768 chiều) | **`"v2"`** — chất lượng tốt hơn, luôn dùng v2 |

---

#### Tham số về cao độ (F0 — pitch)

| Tham số | Giải thích | Nên chọn gì? |
|---------|-----------|----------------|
| `if_f0` | `True` = train model có thể giữ cao độ gốc. `False` = bỏ qua cao độ | **`True`** cho hát, `False` cho chỉ nói |
| `f0_method` | Thuật toán phân tích cao độ: `"rmvpe"`, `"harvest"`, `"pm"`, `"crepe"` | **`"rmvpe"`** — chính xác nhất, nhanh, khuyến nghị |

**So sánh f0_method:**

| Phương pháp | Tốc độ | Chính xác | Ghi chú |
|-------------|--------|-----------|--------|
| `rmvpe` | ★★★★ | ★★★★★ | Tốt nhất, cần `rmvpe.pt` |
| `harvest` | ★★ | ★★★★ | Chậm nhưng ổn định |
| `pm` | ★★★★★ | ★★★ | Nhanh nhất, kém chính xác hơn |
| `crepe` | ★★ | ★★★★ | Cần GPU, chậm |

---

#### Tham số về tài nguyên máy tính

| Tham số | Giải thích | Ví dụ |
|---------|-----------|-------|
| `num_processes` | Số CPU core cho bước preprocess và trích F0 | `4` (hoặc số core bạn có) |
| `gpu_devices_train` | ID GPU dùng để train. `"0"` = GPU đầu tiên. `"0-1"` = 2 GPU | `"0"` |

---

#### Tham số huấn luyện (ảnh hưởng trực tiếp đến chất lượng model)

| Tham số | Giải thích | Gợi ý |
|---------|-----------|-------|
| `total_epochs` | Tổng số vòng lặp train. Nhiều hơn = học tốt hơn (đến giới hạn) | 100–300 epoch cho dataset nhỏ. 50 epoch để test nhanh |
| `save_every_epoch` | Lưu checkpoint G/D sau mỗi N epoch | `10` — lưu để có thể dùng checkpoint giữa chừng |
| `batch_size` | Số mẫu train mỗi lần. Tăng nếu còn VRAM | RTX 3050 (4GB): dùng `4`. RTX 3090 (24GB): có thể dùng `16-32` |

**Gợi ý `batch_size` theo VRAM:**

| VRAM | Batch size |
|------|------------|
| ≤ 4 GB | 2–4 |
| 6–8 GB | 4–8 |
| ≥ 10 GB | 8–16 |

---

#### Tham số về FAISS Index (tùy chọn)

| Tham số | Giải thích | Nên chọn gì? |
|---------|-----------|----------------|
| `skip_index` | `True` = bỏ qua bước tạo FAISS index. `False` = tạo index | **`False`** — nên tạo index vì giúp infer tự nhiên hơn |

---

#### Tham số đặt tên model xuất ra

| Tham số | Giải thích | Ví dụ |
|---------|-----------|-------|
| `infer_weight_name` | Tên file `.pth` nhỏ xuất ra trong `assets/weights/` | `"giong_A_infer"` → tạo `assets/weights/giong_A_infer.pth` |
| `speaker_id` | ID speaker trong filelist, thường `0` khi train 1 giọng | `0` |

### 3.2 — Đặt tham số (SỬA Ô NÀY)

> **Chỉ cần sửa phần `# SỬA` — những phần còn lại có thể giữ nguyên cho lần chạy đầu tiên.**

In [ ]:
from pathlib import Path
from training_pipeline.params import TrainingParams

# ================================================================
# PHẦN CẦN SỬA — Đặt tham số cho thí nghiệm của bạn
# ================================================================

p = TrainingParams(

    # --- Tên thí nghiệm và đường dẫn dataset ---
    experiment_name = "giong_A",              # SỬA: tên thư mục output (logs/tên/)
    trainset_dir    = "datasets/giong_cua_toi",  # SỬA: thư mục chứa .wav (từ Bước 1)

    # --- Sample rate và phiên bản model ---
    sample_rate_label = "40k",  # "32k" | "40k" | "48k"
                                # Khuyến nghị: "40k" cho hầu hết trường hợp
    version = "v2",             # "v1" | "v2" — Luôn dùng "v2"

    # --- Cao độ F0 ---
    if_f0     = True,      # True = train model giữ cao độ (cần cho hát)
                           # False = bỏ cao độ (dùng khi chỉ train giọng nói)
    f0_method = "rmvpe",   # "rmvpe" (tốt nhất) | "harvest" | "pm" | "crepe"

    # --- Tài nguyên máy tính ---
    num_processes    = 4,   # Số CPU core cho preprocess + trích F0
                            # Đặt bằng số core thực tế (thường 4-8)
    gpu_devices_train = "0", # GPU train: "0" = GPU đầu tiên
                             # "0-1" nếu có 2 GPU

    # --- Tham số huấn luyện ---
    total_epochs    = 100,  # Tổng epoch. 50 để test nhanh, 200-300 để chất lượng tốt
    save_every_epoch = 10,  # Lưu checkpoint mỗi N epoch
    batch_size      = 4,    # RTX 3050: 4, GPU mạnh hơn: 8-16

    # --- FAISS Index ---
    skip_index = False,     # False = tạo index sau train (khuyến nghị)
                            # True = bỏ qua (nhanh hơn, nhưng infer kém hơn)

    # --- Xuất model nhỏ ---
    infer_weight_name = "giong_A_infer",  # Tên file .pth xuất ra assets/weights/
    extract_infer_pth = False,             # Giữ False, ta sẽ xuất thủ công ở Bước 9

    # --- Speaker ID ---
    speaker_id = 0,  # 0 khi train 1 giọng (không cần thay đổi)
)

print("✅ Tham số đã đặt xong.")
print("   Tiếp theo: chạy ô 3.3 để xem tóm tắt và kiểm tra.")

### 3.3 — Xem tóm tắt tham số và kiểm tra dataset

In [ ]:
from pathlib import Path
from training_pipeline import steps as train_steps

print("=" * 55)
print("TÓM TẮT THAM SỐ TRAINING")
print("=" * 55)
print(f"  Thí nghiệm   : {p.experiment_name}")
print(f"  Output dir   : logs/{p.experiment_name}/")
print(f"  Dataset dir  : {p.trainset_dir}")
print(f"  Sample rate  : {p.sample_rate_label}")
print(f"  Version      : {p.version}")
print(f"  Có F0 (pitch): {p.if_f0}")
print(f"  F0 method    : {p.f0_method}")
print(f"  CPU processes: {p.num_processes}")
print(f"  GPU train    : {p.gpu_devices_train}")
print(f"  Epochs       : {p.total_epochs}")
print(f"  Save mỗi     : {p.save_every_epoch} epoch")
print(f"  Batch size   : {p.batch_size}")
print(f"  FAISS index  : {'BỎ QUA' if p.skip_index else 'SẼ TẠO'}")
print(f"  Model infer  : assets/weights/{p.infer_weight_name}.pth")
print("=" * 55)

# Kiểm tra dataset
ts = Path(p.trainset_dir)
if not ts.is_dir():
    print(f"\n❌ Thư mục dataset không tồn tại: {ts.resolve()}")
    print("   Tạo thư mục và copy .wav vào (Bước 1.2)")
else:
    wavs = list(ts.glob("*.wav")) + list(ts.glob("*.WAV"))
    if not wavs:
        print(f"\n❌ Không có file .wav trong {ts.resolve()}")
    else:
        print(f"\n✅ Dataset: {len(wavs)} file .wav")

# Kiểm tra mute templates
mm = train_steps.check_mute_template(STANDALONE_ROOT)
if mm:
    print(f"\n❌ logs/mute/ thiếu: {mm}")
    print("   Copy thư mục mute từ bản RVC gốc vào logs/mute/")
else:
    print("✅ logs/mute/: OK")

print("\n→ Nếu tất cả ✅, chạy BƯỚC 4 (Preprocess).")

---
# BƯỚC 4: Preprocess — Cắt và Chuẩn Hóa Audio

**Mục đích:** Đọc file .wav gốc → cắt đoạn im lặng dài → chia thành clip ngắn (~3-5 giây) → chuẩn hóa âm lượng → lưu 2 bản:
- `logs/<tên>/0_gt_wavs/` — audio ở sample rate bạn chọn (40kHz) → **Ground Truth dùng để train**
- `logs/<tên>/1_16k_wavs/` — audio ở 16kHz → **Dùng cho Hubert ở Bước 6**

**Thời gian:** Thường 1-5 phút tùy dung lượng dataset.

**Log:** `logs/<tên>/preprocess.log`

In [ ]:
print("Bắt đầu Preprocess...")
print(f"  Input : {p.trainset_dir}")
print(f"  Output: logs/{p.experiment_name}/0_gt_wavs/ và 1_16k_wavs/")
print()

train_steps.step_preprocess(STANDALONE_ROOT, config, p)

print("\n✅ Bước 4 (Preprocess) hoàn thành.")

### 4.1 — Kiểm tra kết quả Preprocess

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

gt_wavs   = list((exp_dir / "0_gt_wavs").glob("*.wav"))   if (exp_dir / "0_gt_wavs").exists()   else []
_16k_wavs = list((exp_dir / "1_16k_wavs").glob("*.wav"))  if (exp_dir / "1_16k_wavs").exists()  else []

print("Kết quả Preprocess:")
print(f"  0_gt_wavs/   : {len(gt_wavs)} file")
print(f"  1_16k_wavs/  : {len(_16k_wavs)} file")

if gt_wavs:
    print(f"\nVí dụ các file clip đã tạo (5 file đầu):")
    for f in sorted(gt_wavs)[:5]:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name}  ({size_kb:.0f} KB)")
    if len(gt_wavs) > 5:
        print(f"  ... và {len(gt_wavs)-5} file khác")

    if len(gt_wavs) < 10:
        print("\n⚠️  Ít clip. Dataset có thể quá ngắn hoặc phần lớn là im lặng.")
    else:
        print(f"\n✅ Preprocess tạo ra {len(gt_wavs)} clip — sẵn sàng cho Bước 5.")
else:
    print("\n❌ Không có file trong 0_gt_wavs/. Xem log:")
    log_file = exp_dir / "preprocess.log"
    if log_file.exists():
        print(log_file.read_text(encoding='utf-8', errors='ignore')[-2000:])
    else:
        print("  Không tìm thấy preprocess.log")

---
# BƯỚC 5: Trích Xuất F0 (Cao Độ / Pitch)

**Chỉ chạy nếu `if_f0=True` (train model có nhánh cao độ).**

**F0 là gì?** Fundamental Frequency — cao độ cơ bản của giọng (nốt nhạc bạn đang hát). Nếu model biết F0, khi infer nó có thể **giữ đúng nốt nhạc** của bài gốc.

**Bước này làm gì:**
- Đọc từng file trong `1_16k_wavs/`
- Phân tích cao độ theo từng frame (~10ms)
- Lưu kết quả dưới dạng `.npy` vào:
  - `logs/<tên>/2a_f0/` — F0 ở dạng Hz
  - `logs/<tên>/2b-f0nsf/` — F0 ở dạng normalized (dùng cho NSF vocoder)

**Thời gian:** Thường 2-10 phút tùy dataset và phương pháp F0.

In [ ]:
if not p.if_f0:
    print("if_f0=False — Bỏ qua Bước 5 (không train nhánh cao độ)")
    print("Chuyển sang Bước 6.")
else:
    print(f"Bắt đầu trích F0 bằng phương pháp: {p.f0_method}")
    print(f"  Input : logs/{p.experiment_name}/1_16k_wavs/")
    print(f"  Output: logs/{p.experiment_name}/2a_f0/ và 2b-f0nsf/")
    print()

    # Lưu ý: step_extract_f0_and_features chạy cả F0 lẫn Hubert cùng nhau
    # Ta gọi ở Bước 6 để đơn giản hóa
    print("⚠️  F0 và Hubert sẽ được trích cùng nhau ở Bước 6.")
    print("   Bước 5 này chỉ để giải thích — không cần chạy code riêng.")
    print()
    print("→ Chuyển sang Bước 6 để trích đồng thời F0 + Hubert features.")

---
# BƯỚC 6: Trích Xuất Đặc Trưng Hubert

**Hubert là gì?** Một mô hình học sâu (của Meta) được train để hiểu "nội dung" của giọng nói — không phải giọng ai, mà là **người đó đang nói/hát gì** (phoneme, âm tiết). Khi infer, Hubert encode giọng nguồn thành vector nội dung, rồi RVC decode ra giọng đích.

**Bước này làm gì:**
- Tải model `assets/hubert/hubert_base.pt` lên GPU
- Encode từng file trong `1_16k_wavs/` thành vector 768 chiều (v2)
- Lưu vào `logs/<tên>/3_feature768/` dạng `.npy`

**⚠️ Bước này cũng chạy luôn F0 nếu `if_f0=True`.**

**Thời gian:** Thường 5-20 phút tùy dataset và GPU.

In [ ]:
print("Bắt đầu trích F0 + Hubert features...")
print(f"  F0 method: {p.f0_method} (nếu if_f0=True)")
print(f"  Hubert   : assets/hubert/hubert_base.pt")
print(f"  Output   : logs/{p.experiment_name}/2a_f0/, 2b-f0nsf/, 3_feature768/")
print()

train_steps.step_extract_f0_and_features(STANDALONE_ROOT, config, p)

print("\n✅ Bước 6 (F0 + Hubert features) hoàn thành.")

### 6.1 — Kiểm tra kết quả

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

feature_dir = "3_feature768" if p.version == "v2" else "3_feature256"

f0_files       = list((exp_dir / "2a_f0").glob("*.npy"))        if (exp_dir / "2a_f0").exists()       else []
f0nsf_files    = list((exp_dir / "2b-f0nsf").glob("*.npy"))     if (exp_dir / "2b-f0nsf").exists()    else []
feature_files  = list((exp_dir / feature_dir).glob("*.npy"))    if (exp_dir / feature_dir).exists()   else []

print("Kết quả Bước 6:")
print(f"  2a_f0/        : {len(f0_files)} file .npy"      + (" (bỏ qua vì if_f0=False)" if not p.if_f0 else ""))
print(f"  2b-f0nsf/     : {len(f0nsf_files)} file .npy"   + (" (bỏ qua vì if_f0=False)" if not p.if_f0 else ""))
print(f"  {feature_dir}/ : {len(feature_files)} file .npy")

gt_wavs = list((exp_dir / "0_gt_wavs").glob("*.wav")) if (exp_dir / "0_gt_wavs").exists() else []

if len(feature_files) == len(gt_wavs) == 0:
    print("\n❌ Không có file feature. Kiểm tra log.")
elif len(feature_files) != len(gt_wavs):
    print(f"\n⚠️  Số feature ({len(feature_files)}) khác số gt_wav ({len(gt_wavs)}).")
    print("   Có thể một số file bị lỗi. Kiểm tra extract_f0_feature.log")
else:
    print(f"\n✅ {len(feature_files)} feature files khớp với {len(gt_wavs)} gt_wavs.")
    print("   Sẵn sàng train (Bước 7).")

---
# BƯỚC 7: Huấn Luyện Model (Train)

**Đây là bước lâu nhất.** Train 2 mạng:
- **Generator (G)**: học cách chuyển (nội dung Hubert + F0 + speaker) → waveform giống giọng đích
- **Discriminator (D)**: học cách phân biệt âm thanh thật/giả → ép G phải giỏi hơn

**Khi nào dừng?** Không có con số cố định. Với RTX 3050 và dataset ~10 phút:
- 100 epoch ≈ 10-12 giờ
- Checkpoint được lưu mỗi `save_every_epoch` epoch tại `logs/<tên>/G_*.pth`
- Bạn có thể thử checkpoint sớm và quyết định dừng

**File output:**
- `logs/<tên>/G_2333333.pth` — checkpoint Generator (số trong tên là global step)
- `logs/<tên>/D_2333333.pth` — checkpoint Discriminator
- `logs/<tên>/train.log` — log quá trình train

### 7.1 — Kiểm tra filelist trước khi train

Bước train sẽ tự tạo `filelist.txt` — danh sách ghép đường dẫn `(wav | feature | f0 | speaker_id)`. Ta kiểm tra trước để đảm bảo không bị lỗi.

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name
feature_dir = "3_feature768" if p.version == "v2" else "3_feature256"

# Kiểm tra các thư mục cần có trước khi train
cac_kiem_tra = [
    ("0_gt_wavs",      "*.wav",  "Ground truth wavs (Bước 4)"),
    ("1_16k_wavs",     "*.wav",  "16kHz wavs (Bước 4)"),
    (feature_dir,      "*.npy",  "Hubert features (Bước 6)"),
]
if p.if_f0:
    cac_kiem_tra += [
        ("2a_f0",    "*.npy",  "F0 raw (Bước 6)"),
        ("2b-f0nsf", "*.npy",  "F0 NSF (Bước 6)"),
    ]

print("Kiểm tra dữ liệu trước khi train:\n")
tong_ok = True
for ten_thu_muc, pattern, mo_ta in cac_kiem_tra:
    d = exp_dir / ten_thu_muc
    files = list(d.glob(pattern)) if d.exists() else []
    ok = len(files) > 0
    icon = "✓" if ok else "✗"
    print(f"  {icon}  {ten_thu_muc:15s} {len(files):4d} file  ({mo_ta})")
    if not ok:
        tong_ok = False

print()
if tong_ok:
    print("✅ Tất cả dữ liệu sẵn sàng. Có thể train.")
else:
    print("❌ Thiếu dữ liệu. Chạy lại bước bị thiếu trước khi train.")

### 7.2 — Chạy Training

> **⚠️ Ô này chạy rất lâu (nhiều giờ).** Kernel sẽ bị block trong suốt thời gian train.
> 
> Bạn có thể theo dõi tiến trình bằng cách mở file `logs/<tên>/train.log` trong một terminal khác:
> ```
> tail -f logs/giong_A/train.log
> ```

In [ ]:
print("Bắt đầu Training...")
print(f"  Experiment : {p.experiment_name}")
print(f"  Epochs     : {p.total_epochs}")
print(f"  Batch size : {p.batch_size}")
print(f"  GPU        : {p.gpu_devices_train}")
print(f"  Checkpoint : mỗi {p.save_every_epoch} epoch")
print(f"  Log file   : logs/{p.experiment_name}/train.log")
print()
print("⏳ Đang train... (kernel sẽ block cho đến khi xong)")

train_steps.step_train(STANDALONE_ROOT, config, p)

print("\n✅ Bước 7 (Training) hoàn thành.")

### 7.3 — Xem log train và các checkpoint đã lưu

Chạy ô này sau khi train xong (hoặc trong khi train từ terminal khác) để xem tiến trình.

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

# Xem 40 dòng cuối của train.log
train_log = exp_dir / "train.log"
if train_log.exists():
    lines = train_log.read_text(encoding='utf-8', errors='ignore').splitlines()
    print(f"=== train.log (40 dòng cuối, tổng {len(lines)} dòng) ===")
    for line in lines[-40:]:
        print(line)
else:
    print("train.log chưa có — train chưa chạy hoặc chưa bắt đầu ghi.")

print()

# Liệt kê các checkpoint đã lưu
g_checkpoints = sorted(exp_dir.glob("G_*.pth"))
d_checkpoints = sorted(exp_dir.glob("D_*.pth"))

if g_checkpoints:
    print(f"Checkpoints G đã lưu ({len(g_checkpoints)} file):")
    for f in g_checkpoints:
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name}  ({size_mb:.0f} MB)")
else:
    print("Chưa có checkpoint G nào.")

---
# BƯỚC 8: Tạo FAISS Index

**FAISS Index dùng để làm gì?** Khi infer, bên cạnh việc dùng Hubert feature, RVC có thể dùng **retrieval** — tìm vector feature gần nhất trong tập training để blend vào. Điều này giúp giọng infer **tự nhiên hơn** và giống giọng đích hơn.

**Bước này làm gì:**
- Gom tất cả vector từ `3_feature768/`
- Nếu quá nhiều vector: gộp bằng KMeans
- Train FAISS index (IVF + Flat)
- Lưu 2 file:
  - `logs/<tên>/trained_IVF...index` — trung gian (không dùng)
  - `logs/<tên>/added_IVF...index` — **file này dùng khi infer** (copy vào `assets/indices/`)

**Bỏ qua nếu:** `skip_index=True` — khi infer đặt `index_rate=0` thì không cần index.

**Thời gian:** Thường nhanh, 1-5 phút.

In [ ]:
if p.skip_index:
    print("skip_index=True — Bỏ qua Bước 8.")
    print("Khi infer, đặt index_rate=0 để không dùng retrieval.")
else:
    print("Bắt đầu tạo FAISS index...")
    print(f"  Input : logs/{p.experiment_name}/3_feature768/")
    print(f"  Output: logs/{p.experiment_name}/added_IVF...index")
    print()

    for line in train_steps.step_train_index(STANDALONE_ROOT, config, p):
        print(line)

    print("\n✅ Bước 8 (FAISS Index) hoàn thành.")

### 8.1 — Kiểm tra file index

In [ ]:
from pathlib import Path

if p.skip_index:
    print("Đã bỏ qua bước index.")
else:
    exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

    # Tìm file index
    added_indexes  = list(exp_dir.glob("added_*.index"))
    trained_indexes = list(exp_dir.glob("trained_*.index"))

    print("File index đã tạo:")
    for f in trained_indexes:
        size_mb = f.stat().st_size / 1024**2
        print(f"  (trung gian) {f.name}  ({size_mb:.1f} MB)")
    for f in added_indexes:
        size_mb = f.stat().st_size / 1024**2
        print(f"  ✅ (DÙNG ĐỂ INFER) {f.name}  ({size_mb:.1f} MB)")

    if not added_indexes:
        print("❌ Không tìm thấy added_*.index — bước tạo index có thể bị lỗi.")
    else:
        print(f"\n✅ Index sẵn sàng. Copy file added_*.index vào assets/indices/ để dùng khi infer.")

---
# BƯỚC 9: Xuất Model Nhỏ cho Inference

**Tại sao cần xuất?** File `G_*.pth` từ bước train rất nặng (~200-300MB) và chứa cả optimizer state. Để infer, ta chỉ cần **trọng số mạng G** — không cần optimizer. Bước này tạo file `.pth` nhỏ hơn (thường ~60MB) trong `assets/weights/`.

**File output:** `assets/weights/<infer_weight_name>.pth`

> Bạn có thể chọn checkpoint G cụ thể (ví dụ epoch 50) hoặc để trống để lấy checkpoint mới nhất.

### 9.1 — Chọn checkpoint để xuất

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name
g_checkpoints = sorted(exp_dir.glob("G_*.pth"))

if not g_checkpoints:
    print("❌ Chưa có checkpoint G nào. Chạy Bước 7 trước.")
else:
    print(f"Các checkpoint G trong logs/{p.experiment_name}/:\n")
    for i, f in enumerate(g_checkpoints):
        size_mb = f.stat().st_size / 1024**2
        tag = " ← MỚI NHẤT" if i == len(g_checkpoints) - 1 else ""
        print(f"  [{i}] {f.name}  ({size_mb:.0f} MB){tag}")
    print()
    print("→ Mặc định sẽ xuất checkpoint MỚI NHẤT.")
    print("  Nếu muốn xuất checkpoint cụ thể, đặt:")
    print(f'     p.g_checkpoint_for_extract = "{g_checkpoints[0]}"  # đổi tên file')

### 9.2 — Xuất model

In [ ]:
# Đặt checkpoint muốn xuất (để trống = tự chọn mới nhất)
p.g_checkpoint_for_extract = ""   # Hoặc đặt đường dẫn cụ thể
p.infer_weight_name = getattr(p, "infer_weight_name", "giong_A_infer")

print(f"Xuất model nhỏ từ checkpoint {'mới nhất' if not p.g_checkpoint_for_extract else p.g_checkpoint_for_extract}")
print(f"Output: assets/weights/{p.infer_weight_name}.pth")
print()

result = train_steps.step_extract_small_weights(STANDALONE_ROOT, p)
print(result)

# Kiểm tra file đã xuất
output_file = STANDALONE_ROOT / "assets" / "weights" / f"{p.infer_weight_name}.pth"
if output_file.exists():
    size_mb = output_file.stat().st_size / 1024**2
    print(f"\n✅ Model infer đã xuất: {output_file}  ({size_mb:.1f} MB)")
else:
    print("\n❌ Không tìm thấy file output. Kiểm tra lỗi ở trên.")

---
# TỔNG KẾT — Các File Output Sau Khi Train Xong

In [ ]:
from pathlib import Path

exp_dir = STANDALONE_ROOT / "logs" / p.experiment_name

print("=" * 60)
print("TỔNG KẾT CÁC FILE OUTPUT")
print("=" * 60)

# 1. Model weights (cho inference)
weights_dir = STANDALONE_ROOT / "assets" / "weights"
weights = list(weights_dir.glob("*.pth")) if weights_dir.exists() else []
print("\n[1] Model cho inference (assets/weights/):")
for f in weights:
    size_mb = f.stat().st_size / 1024**2
    print(f"    ✅ {f.name}  ({size_mb:.1f} MB)")
if not weights:
    print("    (chưa có — chạy Bước 9)")

# 2. Checkpoints
g_ckpts = sorted(exp_dir.glob("G_*.pth"))
print(f"\n[2] Checkpoints G (logs/{p.experiment_name}/):")
for f in g_ckpts:
    size_mb = f.stat().st_size / 1024**2
    print(f"    {f.name}  ({size_mb:.0f} MB)")
if not g_ckpts:
    print("    (chưa có — chạy Bước 7)")

# 3. FAISS Index
indexes = list(exp_dir.glob("added_*.index"))
print(f"\n[3] FAISS Index (logs/{p.experiment_name}/):")
for f in indexes:
    size_mb = f.stat().st_size / 1024**2
    print(f"    ✅ {f.name}  ({size_mb:.1f} MB)")
if not indexes:
    print("    (chưa có — chạy Bước 8 hoặc skip_index=False)")

print("\n" + "=" * 60)
print("Để INFER (chuyển giọng), bạn cần:")
print(f"  • Model  : assets/weights/{p.infer_weight_name}.pth")
if indexes:
    print(f"  • Index  : {indexes[-1].name} (copy vào assets/indices/)")
print("  • Notebook: RVC_infer_notebook.ipynb")
print("=" * 60)